<a href="https://colab.research.google.com/github/Phoenix2433/Brazilian_e-commerce/blob/main/Brazilian_e_commerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
customers=pd.read_csv('olist_customers_dataset.csv')
orders=pd.read_csv('olist_orders_dataset.csv')
order_items=pd.read_csv('olist_order_items_dataset.csv')
payments=pd.read_csv('olist_order_payments_dataset.csv')
products=pd.read_csv('olist_products_dataset.csv')
reviews=pd.read_csv('olist_order_reviews_dataset.csv')
translation= pd.read_csv('product_category_name_translation.csv')
for name, df in [('customers',customers), ('orders',orders), ('order_items',order_items), ('payments',payments), ('products',products), ('reviews',reviews)]:
  print(f"---{name} ---")
  print(df.columns.tolist())
  print()




---customers ---
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

---orders ---
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

---order_items ---
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

---payments ---
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

---products ---
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

---reviews ---
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']



##Data Merging

In [2]:
merged=orders.merge(customers, on='customer_id', how='left')
merged=merged.merge(order_items, on='order_id', how='left')
merged=merged.merge(products, on='product_id', how='left')
merged=merged.merge(payments, on='order_id', how='left')
merged=merged.merge(reviews, on='order_id', how='left')
print(merged.shape)

(119143, 36)


In [3]:
merged.isnull().sum().sort_values(ascending=False).head(15)

,0
review_comment_title,105154
review_comment_message,68898
order_delivered_customer_date,3421
product_category_name,2542
product_photos_qty,2542
product_description_lenght,2542
product_name_lenght,2542
order_delivered_carrier_date,2086
review_creation_date,997
review_answer_timestamp,997


##Data Cleaning

In [4]:
date_cols= ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
           'order_delivered_customer_date', 'order_estimated_delivery_date'
]
for col in date_cols:
  merged[col] =pd.to_datetime(merged[col], errors='coerce')
merged['is_delivered']= merged['order_delivered_customer_date'].notna()
merged['delivery_days']= ( merged['order_delivered_customer_date']- merged['order_purchase_timestamp']).dt.days
merged['product_category_name']= merged['product_category_name'].fillna('unknown')
merged=merged.rename(columns={"product_name_lenght": "product_name_length","product_description_lenght":"product_description_length"})
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
  merged[col]=merged[col].fillna(merged[col].median())
for col in ['product_name_length', 'product_description_length', 'product_photos_qty']:
  merged[col] = merged[col].fillna(0)


In [5]:
merged.duplicated().sum()
print(merged.shape)
print(merged['order_id'].nunique())

(119143, 38)
99441


In [6]:
merged['review_comment_title']= merged['review_comment_title'].fillna('')
merged['review_comment_message']= merged['review_comment_message'].fillna('')

In [7]:
merged.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]
customer_unique_id,object
customer_zip_code_prefix,int64


In [8]:
merged['review_creation_date']= pd.to_datetime(merged['review_creation_date'], errors='coerce')
merged['review_answer_timestamp']= pd.to_datetime(merged['review_answer_timestamp'], errors='coerce')

In [9]:
merged = merged.merge(translation, on='product_category_name', how='left')
merged['product_category_name_english'] =merged['product_category_name_english'].fillna('unknown')


In [10]:
merged[['product_category_name', 'product_category_name_english']].drop_duplicates().head(10)

,product_category_name,product_category_name_english
0,utilidades_domesticas,housewares
3,perfumaria,perfumery
4,automotivo,auto
5,pet_shop,pet_shop
6,papelaria,stationery
8,unknown,unknown
10,moveis_decoracao,furniture_decor
11,moveis_escritorio,office_furniture
13,ferramentas_jardim,garden_tools
15,informatica_acessorios,computers_accessories


##Q1: Delivery Performance vs. Customer Satisfaction(Pandas)

In [11]:
delivered_filter= merged[merged['is_delivered']]
delivered_filter= merged[merged['is_delivered']].copy()
delivered_filter['is_late']= delivered_filter['order_delivered_customer_date']> delivered_filter['order_estimated_delivery_date']
delivered_filter.groupby('is_late')['review_score'].mean()

,review_score
is_late,
False,4.208608
True,2.546542


##Q2: Revenue by Category vs. Review Score(Pandas)

In [12]:
category_summary = delivered_filter.groupby('product_category_name_english').agg({'price': 'sum', 'review_score': 'mean', 'order_id': 'count'}).sort_values('price', ascending=False)
category_summary = category_summary.rename(columns={'order_id': 'order_count'})
print(category_summary)

                                    price  review_score  order_count
product_category_name_english                                       
health_beauty                  1275856.38      4.185295         9818
watches_gifts                  1214342.45      4.069320         6075
bed_bath_table                 1092461.20      3.915276        11814
sports_leisure                  995823.26      4.164148         8791
computers_accessories           926446.44      3.988897         7962
...                                   ...           ...          ...
flowers                           1110.04      4.419355           33
home_comfort_2                     773.17      3.642857           31
cds_dvds_musicals                  730.00      4.642857           14
fashion_childrens_clothes          519.95      5.000000            7
security_and_services              283.29      2.500000            2

[72 rows x 3 columns]


##Q3: Payment Method Analysis(Pandas)

In [13]:
payment_count=merged['payment_type'].value_counts()
print("Payment Type Frequency:")
print(payment_count)
print()
payment_avg_value = merged.groupby('payment_type')['price'].mean().sort_values(ascending=False)
print(payment_avg_value)
print()
installment_corr = merged['price'].corr(merged['payment_installments'])
print(f"Correlation between price and installments: {installment_corr:.3f}")

Payment Type Frequency:
payment_type
credit_card    87776
boleto         23190
voucher         6465
debit_card      1706
not_defined        3
Name: count, dtype: int64

payment_type
credit_card    126.268956
debit_card     108.507540
voucher        105.061521
boleto         104.526208
not_defined           NaN
Name: price, dtype: float64

Correlation between price and installments: 0.278


##Q4: Customer Geography & Spend(Pandas)

In [14]:
geography_summary= merged.groupby('customer_state').agg({'price':['sum', 'mean'], 'order_id': 'count'})
geography_summary.columns = ['total_revenue', 'avg_order_value', 'order_count']
geography_summary = geography_summary.sort_values('total_revenue', ascending=False)
print("Revenue and Order Value by State:")
print(geography_summary)

Revenue and Order Value by State:
                total_revenue  avg_order_value  order_count
customer_state                                             
SP                 5477008.74       109.836734        50265
RJ                 1921752.53       124.586874        15518
MG                 1645847.27       119.977203        13819
RS                  791716.84       121.094653         6573
PR                  708794.22       118.369108         6043
BA                  543243.99       133.540804         4091
SC                  539896.20       125.004909         4345
DF                  315122.29       126.048916         2516
GO                  313198.27       127.679686         2466
ES                  284771.30       121.127733         2360
PE                  272271.97       143.225655         1906
CE                  240095.13       154.302783         1565
PA                  184889.93       164.639297         1129
MT                  171109.74       151.693032         1132
PB    

##Q5: Order Fulfillment Health(Pandas)

In [15]:
order_status_pct=merged['order_status'].value_counts(normalize=True) * 100
print("Order Status Breakdown (%):")
print(order_status_pct)
print()
merged['is_problem']= merged['order_status'].isin(['unavailable','canceled'])
state_problem_pct= merged.groupby('customer_state')['is_problem'].mean().sort_values(ascending=False) * 100
print("Problem Order Rate by State (%):")
print(state_problem_pct)
print()
category_problem_pct =merged.groupby('product_category_name_english')['is_problem'].mean().sort_values(ascending=False) * 100
print("Problem Order Rate by Category(%):")
print(category_problem_pct)
unknown_check = merged[merged['product_category_name_english']=='unknown']['is_problem'].value_counts()
print("Sample size behind 'unknown' category problem rate:")
print(unknown_check)

Order Status Breakdown (%):
order_status
delivered      97.129500
shipped         1.054195
canceled        0.629496
unavailable     0.547242
invoiced        0.317266
processing      0.315587
created         0.004197
approved        0.002518
Name: proportion, dtype: float64

Problem Order Rate by State (%):
customer_state
RO    2.397260
RR    1.923077
SE    1.488834
SP    1.406545
MA    1.285047
PR    1.274202
MG    1.165063
RJ    1.076170
PI    1.041667
GO    0.973236
DF    0.953895
PB    0.931677
BA    0.928868
SC    0.920598
RS    0.882398
CE    0.830671
MS    0.812065
ES    0.805085
PA    0.620018
TO    0.588235
AM    0.578035
PE    0.524659
RN    0.522648
MT    0.441696
AL    0.431034
AP    0.000000
AC    0.000000
Name: is_problem, dtype: float64

Problem Order Rate by Category(%):
product_category_name_english
unknown                                  32.800935
dvds_blu_ray                              4.225352
diapers_and_hygiene                       2.564103
construction_tools_s

In [16]:
import sqlite3
conn = sqlite3.connect('olist.db')
merged.to_sql('orders_merged', conn, if_exists='replace', index=False)
conn.close()


In [17]:
conn = sqlite3.connect('olist.db')

##Q1: Delivery Performance vs. Customer Satisfaction(SQL)

In [18]:
query="""
SELECT CASE WHEN order_delivered_customer_date> order_estimated_delivery_date THEN 'Late'
            ELSE 'On Time'
       END AS delivery_status,
       AVG(review_score) AS avg_review_score, COUNT(*) AS order_count
FROM orders_merged
WHERE order_delivered_customer_date IS NOT NULL
GROUP BY delivery_status;
"""
result= pd.read_sql(query, conn)
print(result)

  delivery_status  avg_review_score  order_count
0            Late          2.546542         9068
1         On Time          4.208608       106654


##Q2: Revenue by Category vs. Review Score(SQL)

In [19]:
query="""
SELECT product_category_name_english,SUM(price) AS Price, AVG(review_score) AS review_score, COUNT(order_id) AS order_count
FROM orders_merged
WHERE order_status = 'delivered'
GROUP BY product_category_name_english
ORDER BY price DESC;
"""
result=pd.read_sql(query, conn)
print(result)


   product_category_name_english       Price  review_score  order_count
0                  health_beauty  1275776.49      4.185333         9816
1                  watches_gifts  1214620.45      4.069629         6077
2                 bed_bath_table  1092461.20      3.915276        11814
3                 sports_leisure   995980.76      4.164606         8791
4          computers_accessories   926557.43      3.988520         7963
..                           ...         ...           ...          ...
67                       flowers     1110.04      4.419355           33
68                home_comfort_2      773.17      3.642857           31
69             cds_dvds_musicals      730.00      4.642857           14
70     fashion_childrens_clothes      519.95      5.000000            7
71         security_and_services      283.29      2.500000            2

[72 rows x 4 columns]


##Q3: Payment Method Analysis(SQL)

In [24]:
query="""
SELECT payment_type,COUNT(*) AS payment_count, AVG(price) AS payment_avg_value
FROM orders_merged
GROUP BY payment_type
ORDER BY payment_count DESC;
"""
result=pd.read_sql(query,conn)
print(result)

  payment_type  payment_count  payment_avg_value
0  credit_card          87776         126.268956
1       boleto          23190         104.526208
2      voucher           6465         105.061521
3   debit_card           1706         108.507540
4  not_defined              3                NaN
5         None              3          44.990000


Note: Payment type counts and average order values match exactly between pandas and SQL. The installment vs price correlation was calculated only in pandas, since SQLite does not have a built in correlation function.

##Q4: Customer Geography & Spend(SQL)

In [27]:
query="""
SELECT customer_state, SUM(price) AS total_revenue, AVG(price) AS avg_order_value, COUNT(price) AS order_count
FROM orders_merged
GROUP BY customer_state
ORDER BY total_revenue DESC;
"""
pd.set_option('display.float_format','{:.2f}'.format)
result= pd.read_sql(query, conn)
print(result)

   customer_state  total_revenue  avg_order_value  order_count
0              SP     5477008.74           109.84        49865
1              RJ     1921752.53           124.59        15425
2              MG     1645847.27           119.98        13718
3              RS      791716.84           121.09         6538
4              PR      708794.22           118.37         5988
5              BA      543243.99           133.54         4068
6              SC      539896.20           125.00         4319
7              DF      315122.29           126.05         2500
8              GO      313198.27           127.68         2453
9              ES      284771.30           121.13         2351
10             PE      272271.97           143.23         1901
11             CE      240095.13           154.30         1556
12             PA      184889.93           164.64         1123
13             MT      171109.74           151.69         1128
14             PB      123816.24           193.46      

##Q5: Order Fulfillment Health(SQL)

In [28]:
query1="""
SELECT order_status, COUNT(*) AS order_count,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders_merged), 2) AS pct
FROM orders_merged
GROUP BY order_status
ORDER BY order_count DESC;
"""
print("Order Status Breakdown:")
print(pd.read_sql(query1, conn))
print()

query2="""
SELECT customer_state,
       ROUND(100.0 * SUM(CASE WHEN order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) / COUNT(*), 2) AS problem_pct,
       COUNT(*) AS order_count
FROM orders_merged
GROUP BY customer_state
ORDER BY problem_pct DESC;
"""
print("Problem Order Rate by State:")
print(pd.read_sql(query2, conn))
print()

query3="""
SELECT product_category_name_english,
       ROUND(100.0 * SUM(CASE WHEN order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) / COUNT(*), 2) AS problem_pct,
       COUNT(*) AS order_count
FROM orders_merged
GROUP BY product_category_name_english
ORDER BY problem_pct DESC;
"""
print("Problem Order Rate by Category:")
print(pd.read_sql(query3, conn))


Order Status Breakdown:
  order_status  order_count   pct
0    delivered       115723 97.13
1      shipped         1256  1.05
2     canceled          750  0.63
3  unavailable          652  0.55
4     invoiced          378  0.32
5   processing          376  0.32
6      created            5  0.00
7     approved            3  0.00

Problem Order Rate by State:
   customer_state  problem_pct  order_count
0              RO         2.40          292
1              RR         1.92           52
2              SE         1.49          403
3              SP         1.41        50265
4              MA         1.29          856
5              PR         1.27         6043
6              MG         1.17        13819
7              RJ         1.08        15518
8              PI         1.04          576
9              GO         0.97         2466
10             DF         0.95         2516
11             PB         0.93          644
12             BA         0.93         4091
13             SC       

## Exporting Data for Power BI

In [32]:
columns_needed = ['order_id', 'customer_state', 'customer_city', 'order_status', 'is_delivered',
                  'delivery_days', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date',
                  'product_category_name_english', 'price', 'payment_type', 'payment_installments', 'payment_value', 'review_score']
merged_slim = merged[columns_needed].copy()
merged['is_late'] = merged['order_delivered_customer_date']> merged['order_estimated_delivery_date']
merged_slim['country'] ='Brazil'
merged_slim.to_csv('merged_slim.csv', index=False)
category_summary.to_csv('category_summary.csv')
geography_summary.to_csv('geography_summary.csv')

##Summary:
Dataset: Brazilian e-commerce orders(Olist/Kaggle), 6 tables merged, 119K rows.

Tools: pandas, SQL(SQLite)

Key Findings:

Q1: Delivery vs Satisfaction: Late deliveries average 2.55 stars vs 4.21 stars for on time deliveries, which shows a steep satisfaction drop.

Q2: Category Revenue vs Rating: health_beauty leads in both revenue and rating. bed_bath_table is on 3rd spot in revenue but underperforms on satisfaction(only 3.92 stars).

Q3: Payment Methods: Credit card dominates. Installments weakly correlate with price(r=0.28).

Q4: Geography: SP drives the most orders and revenue but has the lowest avg. orders value.

Q5: Fulfillment Health: 97.1% delivery success rate. No major state/category red flags, except orders with missing category data(32.8% problem rate, a data quality issue).